# TradeSense · Component 03 · Audit of Layer 1 versions and of the three frozen model packages (read only)

**Why.** Two new facts came out of the data audit: (1) the daily Layer 1 files do not match their saved hashes, and (2) the stored `volume_acceleration` still contains the extreme warm-up values (about -7173 for BTC and -6435 for ETH) although the correction report says it was stabilised. The three frozen models (Autoencoder v2, XGBoost v1, Transformer v1) were trained on Layer 1 bars, so we must know **which version of the data they saw** and whether their scalers were distorted by those values.

**Part A** compares every Layer 1 copy (daily cache, 90-day matrices, the split partitions, the backups before the fix) with each other and with the written contract formula for `volume_acceleration` (ddof=1, floor, previous 15 closed bars, first 15 bars masked), and measures how much two extreme values would distort a scaler fitted on the training split.
**Part B** prints, for each model package, when its files were written, its manifest, preprocessing (scalers), protocol, validation verdict, limitations and the shape of its saved scores.

Nothing is changed. CPU runtime, a few minutes. The full text is saved as `data_audit_v1/model_audit_v1.txt`: **attach that file here**.

In [ ]:
import os, re, json, time, glob, hashlib, warnings, traceback
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir()
OUT = BASE / 'data_audit_v1'; OUT.mkdir(exist_ok=True)
RES = {}; FAILS = []; LINES = []
def say(s=''): print(s); LINES.append(str(s))
def section(name):
    def deco(fn):
        try: fn()
        except Exception as e: FAILS.append(name); say(f'!! section {name} failed: {type(e).__name__}: {str(e)[:200]}'); traceback.print_exc(limit=2)
        return fn
    return deco
def sha(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

Mounted at /content/drive


## Part A · Layer 1 versions and volume_acceleration

In [ ]:
CONTRACT_FLOOR = {'BTCUSDT': 2.539920310163499, 'ETHUSDT': 27.765804660846563}
L1D = BASE / 'processed_layer1_final_v1'
@section('layer1_versions')
def _():
    def load(p): return pd.read_parquet(p)
    res = {}
    for sym in ('BTCUSDT', 'ETHUSDT'):
        s = sym.lower(); M = load(L1D / 'processed_matrices' / f'{s}_l1_feature_matrix_90d.parquet'); va = M['volume_acceleration'].to_numpy(float); say(f'[{sym}] matrix rows {len(M)}; volume_acceleration min {np.nanmin(va):.2f} max {np.nanmax(va):.2f}; |value|>20: {int((np.abs(va) > 20).sum())}; first 18 values: ' + str(np.round(va[:18], 2).tolist()))
        v = M['volume'].astype(float); m15 = v.shift(1).rolling(15).mean(); sd15 = v.shift(1).rolling(15).std(ddof=1); ref = ((v - m15) / np.maximum(sd15, CONTRACT_FLOOR[sym])).to_numpy().copy(); ref[:15] = 0.0; ok = np.arange(len(M)) >= 15
        rel = np.abs(va[ok] - ref[ok]) / np.maximum(np.abs(ref[ok]), 1e-6); say(f'  contract recompute vs stored (after warm-up): max rel err {np.nanmax(rel):.2e}; share within 1e-6: {np.mean(rel < 1e-6):.4f}; stored max |value| after warm-up {np.nanmax(np.abs(va[ok])):.2f}; recomputed max |value| {np.nanmax(np.abs(ref[ok])):.2f}')
        say(f'  first 15 bars masked to 0 in the stored matrix: {bool(np.all(va[:15] == 0))}')
        res[sym] = {'matrix_min': float(np.nanmin(va)), 'first15_masked': bool(np.all(va[:15] == 0)), 'max_rel_err': float(np.nanmax(rel))}
        # distortion of a scaler fitted on the first 70% (train split)
        n_tr = int(len(M) * 0.7); a = va[:n_tr]; b = va[15:n_tr]; say(f'  train split ({n_tr} rows): std with the first 15 bars {np.std(a):.3f} vs without {np.std(b):.3f}; mean {np.mean(a):.3f} vs {np.mean(b):.3f}; robust scale (IQR/1.349) {np.subtract(*np.percentile(a, [75, 25])) / 1.349:.3f}')
        # compare other copies
        parts = {p.name: p for p in (L1D / 'partitions').glob(f'{s}_*.parquet')}; say('  partitions: ' + ', '.join(f'{k} ({len(load(p))} rows)' for k, p in parts.items()))
        for k, p in parts.items():
            P = load(p); n = min(len(P), len(M)); same = bool(np.allclose(P['volume_acceleration'].to_numpy(float)[:n], va[:n], equal_nan=True)) if 'train' in k else None
            if same is not None: say(f'    {k}: volume_acceleration equals the first {n} matrix rows: {same}; min {P["volume_acceleration"].min():.2f}')
        dc = pd.concat([load(p) for p in sorted((L1D / 'daily_cache').glob(f'{sym}_l1_*.parquet'))]).reset_index(drop=True); n = min(len(dc), len(M)); dif = {c: float(np.nanmax(np.abs(dc[c].to_numpy(float)[:n] - M[c].to_numpy(float)[:n]))) for c in M.columns if c != 'datetime'}
        say(f'  daily_cache vs matrix, max abs difference per column: ' + json.dumps({k: float(f'{v:.3g}') for k, v in dif.items()})); say(f'  daily_cache volume_acceleration min {dc["volume_acceleration"].min():.2f}; first 18 values {np.round(dc["volume_acceleration"].to_numpy(float)[:18], 2).tolist()}')
        bk = L1D / 'backups_pre_vol_acc_fix'; bp = [p for p in bk.glob('*.parquet') if p.name.startswith(s)]
        for p in bp[:4]:
            B = load(p); say(f'  backup {p.name}: rows {len(B)}; volume_acceleration min {B["volume_acceleration"].min():.2f} max {B["volume_acceleration"].max():.2f}')
        res[sym]['daily_cache_vs_matrix'] = dif
    RES['layer1_versions'] = res

[BTCUSDT] matrix rows 129600; volume_acceleration min -15.62 max 117.37; |value|>20: 169; first 18 values: [0.0, -15.62, -1.38, -3.93, -1.35, -1.46, -0.98, -1.48, -1.3, -0.71, -0.59, -1.1, -0.04, -0.53, 1.34, -0.38, -0.92, 1.3]
  contract recompute vs stored (after warm-up): max rel err 1.67e-10; share within 1e-6: 1.0000; stored max |value| after warm-up 117.37; recomputed max |value| 117.37
  first 15 bars masked to 0 in the stored matrix: False
  train split (90720 rows): std with the first 15 bars 2.026 vs without 2.025; mean 0.152 vs 0.152; robust scale (IQR/1.349) 0.816
  partitions: btcusdt_train_split_70.parquet (90720 rows), btcusdt_val_split_15.parquet (18720 rows), btcusdt_test_split_15.parquet (20160 rows)
    btcusdt_train_split_70.parquet: volume_acceleration equals the first 90720 matrix rows: True; min -15.62
  daily_cache vs matrix, max abs difference per column: {"open": 0.0, "high": 0.0, "low": 0.0, "close": 0.0, "volume": 0.0, "trades_count": 0.0, "log_returns": 0.0

## Part B · The three frozen model packages

In [ ]:
import datetime as dt
def mt(p): return dt.datetime.fromtimestamp(p.stat().st_mtime, dt.timezone.utc).strftime('%Y-%m-%d %H:%M')
def jshow(p, cap):
    try: o = json.loads(p.read_text()); t = json.dumps(o, indent=1, default=str); say(f'    --- {p.relative_to(p.parents[2]) if len(p.parents) > 2 else p.name} (written {mt(p)})'); say(t[:cap] + (' ...[cut]' if len(t) > cap else ''))
    except Exception as e: say(f'    could not read {p.name}: {e}')
@section('model_packages')
def _():
    for pkg in ('models_autoencoder_v2', 'models_xgboost_v1', 'models_transformer_v1'):
        root = BASE / pkg; say(f'\n######## {pkg}'); files = sorted(p for p in root.rglob('*') if p.is_file()); say(f'  files {len(files)}; newest {max(mt(p) for p in files)}; oldest {min(mt(p) for p in files)}')
        say('  weights/model files: ' + '; '.join(f'{p.relative_to(root)} {mt(p)}' for p in files if p.suffix in ('.pt', '.json') and 'models' in p.parts)[:900])
        jshow(root / 'manifest.json', 2500)
        for p in sorted((root / 'preprocessing').glob('*.json')): jshow(p, 2200)
        for p in sorted((root / 'protocol').glob('*.json')): jshow(p, 2500)
        for p in sorted((root / 'validation').glob('*.json')): jshow(p, 2500)
        lim = root / 'reports' / 'limitations.md'
        if lim.exists(): say(f'    --- reports/limitations.md (written {mt(lim)})'); say(lim.read_text()[:2500])
        say('    reports: ' + ', '.join(f'{p.name}' for p in sorted((root / 'reports').glob('*'))))
        for p in sorted((root / 'scores').glob('*.parquet'))[:6]:
            df = pd.read_parquet(p); num = df.select_dtypes('number'); say(f'    scores/{p.name}: rows {len(df)} cols {list(df.columns)[:10]}' + (f'; {df.iloc[:, 0].min()} -> {df.iloc[:, 0].max()}' if len(df) else ''))
    RES['models_audited'] = True


######## models_autoencoder_v2
  files 47; newest 2026-10-11 12:57; oldest 2026-10-11 12:47
  weights/model files: models/btc/ctx_bar/arch.json 2026-10-11 12:56; models/btc/ctx_bar/seed11.pt 2026-10-11 12:56; models/btc/ctx_bar/seed22.pt 2026-10-11 12:56; models/btc/ctx_bar/seed33.pt 2026-10-11 12:56; models/btc/dn_bar/arch.json 2026-10-11 12:56; models/btc/dn_bar/seed11.pt 2026-10-11 12:56; models/btc/dn_bar/seed22.pt 2026-10-11 12:56; models/btc/dn_bar/seed33.pt 2026-10-11 12:56; models/eth/ctx_bar/arch.json 2026-10-11 12:56; models/eth/ctx_bar/seed11.pt 2026-10-11 12:56; models/eth/ctx_bar/seed22.pt 2026-10-11 12:56; models/eth/ctx_bar/seed33.pt 2026-10-11 12:56; models/eth/dn_bar/arch.json 2026-10-11 12:56; models/eth/dn_bar/seed11.pt 2026-10-11 12:56; models/eth/dn_bar/seed22.pt 2026-10-11 12:56; models/eth/dn_bar/seed33.pt 2026-10-11 12:56; models/selected_model.json 2026-10-11 12:56; models/thresholds_by_budget.json 2026-10-11 12:56
    --- research datasets/models_autoencoder_

## Save

In [ ]:
txt = '\n'.join(LINES); (OUT / 'model_audit_v1.txt').write_text(txt); print(f'\nfailed sections: {FAILS or "none"}; saved {len(txt)} characters to {OUT / "model_audit_v1.txt"}')
print('==== ATTACH data_audit_v1/model_audit_v1.txt (or paste the printed text above) ====')


failed sections: none; saved 39598 characters to /content/drive/MyDrive/research datasets/data_audit_v1/model_audit_v1.txt
==== ATTACH data_audit_v1/model_audit_v1.txt (or paste the printed text above) ====
